In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path



In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "colleges": DATA_DIR / "College-ALL COLLEGE.xlsx",
    "universities": DATA_DIR / "University-ALL UNIVERSITIES.xlsx",
    "standalone": DATA_DIR / "Standalone-ALL STANDALONE.xlsx",
    "research": DATA_DIR / "R & D Institutes.xlsx",
    "vidya_lakshmi": DATA_DIR / "vidya_lakshmiAll.xlsx"
}

datasets = {
    name: pd.read_excel(path, header=2)
    for name, path in FILES.items()
}

for name, df in datasets.items():
    print(f"{name:<15} | Shape: {df.shape}")

colleges        | Shape: (54571, 12)
universities    | Shape: (1438, 7)
standalone      | Shape: (17092, 8)
research        | Shape: (291, 6)
vidya_lakshmi   | Shape: (1452, 9)


In [3]:
for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"DATASET: {name.upper()}")
    print(f"{'=' * 60}")

    for column in df.columns:
        print(column)


DATASET: COLLEGES
Aishe Code
Name
State
District
Website
Year Of Establishment
Location
College Type
Manegement
University Aishe Code
University Name
University Type

DATASET: UNIVERSITIES
Aishe Code
Name
State
District
Website
Year Of Establishment
Location

DATASET: STANDALONE
Aishe Code
Name
State
District
Year Of Establishment
Location
Standalone Type
Manegement

DATASET: RESEARCH
S. No.
AISHE Code
Institute Name
State Name
District Name
Administrative Ministry

DATASET: VIDYA_LAKSHMI
S. No.
AISHE Code
Institute Name
State
District
Management Type
Website
Year Of Establishment
Location


In [4]:
COLUMN_MAPPING = {
    "colleges": {
        "Aishe Code": "aishe_code",
        "Name": "institution_name",
        "State": "state",
        "District": "district",
        "Website": "website",
        "Year Of Establishment": "establishment_year",
        "Location": "location",
        "College Type": "institution_type",
        "Manegement": "management_type",
        "University Aishe Code": "affiliated_university_code",
        "University Name": "affiliated_university_name",
        "University Type": "university_type"
    },
    "universities": {
        "Aishe Code": "aishe_code",
        "Name": "institution_name",
        "State": "state",
        "District": "district",
        "Website": "website",
        "Year Of Establishment": "establishment_year",
        "Location": "location"
    },
    "standalone": {
        "Aishe Code": "aishe_code",
        "Name": "institution_name",
        "State": "state",
        "District": "district",
        "Year Of Establishment": "establishment_year",
        "Location": "location",
        "Standalone Type": "institution_type",
        "Manegement": "management_type"
    },
    "research": {
        "AISHE Code": "aishe_code",
        "Institute Name": "institution_name",
        "State Name": "state",
        "District Name": "district",
        "Administrative Ministry": "administrative_ministry"
    },
    "vidya_lakshmi": {
        "AISHE Code": "aishe_code",
        "Institute Name": "institution_name",
        "State": "state",
        "District": "district",
        "Management Type": "management_type",
        "Website": "website",
        "Year Of Establishment": "establishment_year",
        "Location": "location"
    }
}

In [5]:
standardized_datasets = {}

for name, df in datasets.items():
    temp = df.rename(columns=COLUMN_MAPPING[name]).copy()

    temp["institution_category"] = name.replace("_", " ").title()
    temp["source_dataset"] = name

    standardized_datasets[name] = temp

    print(
        f"{name:<15} | "
        f"Rows: {len(temp):>6} | "
        f"Columns: {len(temp.columns):>2}"
    )

colleges        | Rows:  54571 | Columns: 14
universities    | Rows:   1438 | Columns:  9
standalone      | Rows:  17092 | Columns: 10
research        | Rows:    291 | Columns:  8
vidya_lakshmi   | Rows:   1452 | Columns: 11


In [6]:
for name, df in standardized_datasets.items():
    print(f"\n{name.upper()}")
    print(df.columns.tolist())


COLLEGES
['aishe_code', 'institution_name', 'state', 'district', 'website', 'establishment_year', 'location', 'institution_type', 'management_type', 'affiliated_university_code', 'affiliated_university_name', 'university_type', 'institution_category', 'source_dataset']

UNIVERSITIES
['aishe_code', 'institution_name', 'state', 'district', 'website', 'establishment_year', 'location', 'institution_category', 'source_dataset']

STANDALONE
['aishe_code', 'institution_name', 'state', 'district', 'establishment_year', 'location', 'institution_type', 'management_type', 'institution_category', 'source_dataset']

RESEARCH
['S. No.', 'aishe_code', 'institution_name', 'state', 'district', 'administrative_ministry', 'institution_category', 'source_dataset']

VIDYA_LAKSHMI
['S. No.', 'aishe_code', 'institution_name', 'state', 'district', 'management_type', 'website', 'establishment_year', 'location', 'institution_category', 'source_dataset']


In [7]:
UNIFIED_COLUMNS = [
    "aishe_code",
    "institution_name",
    "institution_category",
    "state",
    "district",
    "website",
    "establishment_year",
    "location",
    "institution_type",
    "management_type",
    "affiliated_university_code",
    "affiliated_university_name",
    "university_type",
    "administrative_ministry",
    "source_dataset"
]

for name, df in standardized_datasets.items():
    standardized_datasets[name] = df.reindex(columns=UNIFIED_COLUMNS)

for name, df in standardized_datasets.items():
    print(f"{name:<15} | Columns: {len(df.columns)}")

colleges        | Columns: 15
universities    | Columns: 15
standalone      | Columns: 15
research        | Columns: 15
vidya_lakshmi   | Columns: 15


In [8]:
combined_df = pd.concat(
    standardized_datasets.values(),
    ignore_index=True
)

print("Combined dataset shape:", combined_df.shape)
print("\nInstitution categories:")
print(combined_df["institution_category"].value_counts())

combined_df.head()

Combined dataset shape: (74844, 15)

Institution categories:
institution_category
Colleges         54571
Standalone       17092
Vidya Lakshmi     1452
Universities      1438
Research           291
Name: count, dtype: int64


,aishe_code,institution_name,institution_category,state,district,website,establishment_year,location,institution_type,management_type,affiliated_university_code,affiliated_university_name,university_type,administrative_ministry,source_dataset
0,C-59634,"061-PT. SRI JAGANNATH PRASAD MAHAVIDHYALAYA,DA...",Colleges,Uttar Pradesh,Agra,WWW.JPCOLLEGE.IN.COM,2016,Rural,Affiliated College,Private Un-Aided,U-0509,"Dr. B. R. Ambedkar University, Agra",State Public University,NaN,colleges
1,C-8913,100002-J.D.M.V.P. SAMAJ'S SHRI S. S. PATIL ART...,Colleges,Maharashtra,Jalgaon,http://www.jdmvpascjal.ac.in,1972,Urban,Affiliated College,Private Aided (Government Aided),U-0320,Kavayitri Bahinabai Chaudhari North Maharashtr...,State Public University,NaN,colleges
2,C-8922,100004-Khandesh College Education Society'S Co...,Colleges,Maharashtra,Jalgaon,www.coe.kces.in,1965,Urban,Affiliated College,Private Aided (Government Aided),U-0320,Kavayitri Bahinabai Chaudhari North Maharashtr...,State Public University,NaN,colleges
3,C-9030,"100005-KCES'S S.S.MANIYAR LAW COLLEGE, JALGAON.",Colleges,Maharashtra,Jalgaon,https://ssmlc.kces.in,1970,Urban,Affiliated College,Private Aided (Government Aided),U-0320,Kavayitri Bahinabai Chaudhari North Maharashtr...,State Public University,NaN,colleges
4,C-9025,100006-KRIDA RASIK EDUCATION SOCIETY INSTITUTE...,Colleges,Maharashtra,Jalgaon,www.sabaheticollege.org,1983,Urban,Affiliated College,Private Aided (Government Aided),U-0320,Kavayitri Bahinabai Chaudhari North Maharashtr...,State Public University,NaN,colleges


In [9]:
missing_summary = pd.DataFrame({
    "missing_count": combined_df.isnull().sum(),
    "missing_percentage": (combined_df.isnull().mean() * 100).round(2)
})

missing_summary.sort_values(
    by="missing_percentage",
    ascending=False
)

,missing_count,missing_percentage
administrative_ministry,74553,99.61
affiliated_university_code,20320,27.15
affiliated_university_name,20273,27.09
university_type,20273,27.09
website,17524,23.41
institution_type,3181,4.25
management_type,1729,2.31
establishment_year,370,0.49
location,329,0.44
district,4,0.01


In [11]:
for column in combined_df.select_dtypes(include="object").columns:
    values = combined_df[column].astype("string").str.strip()

    placeholders = values.isin(["-", "", "NA", "N/A", "None", "nan"])

    if placeholders.sum() > 0:
        print(f"{column:<30} : {placeholders.sum()}")

website                        : 5132
establishment_year             : 2136
location                       : 924
management_type                : 544


C:\Users\moury\AppData\Local\Temp\ipykernel_28576\2732890918.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in combined_df.select_dtypes(include="object").columns:


In [12]:
PLACEHOLDERS = ["-", "", "NA", "N/A", "None", "nan"]

for column in combined_df.select_dtypes(include=["str", "object"]).columns:
    combined_df[column] = combined_df[column].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

    combined_df[column] = combined_df[column].replace(
        PLACEHOLDERS, np.nan
    )

print("Remaining placeholder values:")

for column in combined_df.select_dtypes(include=["str", "object"]).columns:
    count = combined_df[column].isin(PLACEHOLDERS).sum()

    if count > 0:
        print(column, count)

print("\nTotal missing values:", combined_df.isnull().sum().sum())

Remaining placeholder values:

Total missing values: 167292


In [15]:
def normalize_url(url):
    if pd.isna(url):
        return np.nan

    url = str(url).strip()

    if not url:
        return np.nan

    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url

    return url


combined_df["website"] = combined_df["website"].apply(normalize_url)

print("Available websites:", combined_df["website"].notna().sum())
print("Missing websites:", combined_df["website"].isna().sum())

combined_df[["institution_name", "website"]].head(10)

Available websites: 52188
Missing websites: 22656


,institution_name,website
0,"061-PT. SRI JAGANNATH PRASAD MAHAVIDHYALAYA,DA...",https://WWW.JPCOLLEGE.IN.COM
1,100002-J.D.M.V.P. SAMAJ'S SHRI S. S. PATIL ART...,http://www.jdmvpascjal.ac.in
2,100004-Khandesh College Education Society'S Co...,https://www.coe.kces.in
3,"100005-KCES'S S.S.MANIYAR LAW COLLEGE, JALGAON.",https://ssmlc.kces.in
4,100006-KRIDA RASIK EDUCATION SOCIETY INSTITUTE...,https://www.sabaheticollege.org
5,100007-SSBT'S COLLEGE OF ENGINEERING & TECHNOL...,https://www.sscoetjalgaon.ac.in
6,"100008-SES'S COLLEGE OF PHYSICAL EDUCATION, JA...",https://www.bped.jal.org
7,100009- IQRA EDUCATION SOCIETY'S H.J.THIM COLL...,https://www.iqrahjtc.in
8,100010-KCES'S INSTITUTE OF MANAGEMENT & RESEAR...,https://www.imr.ac.in
9,"100043-IES'S IQRA COLLEGE OF EDUCATION, MEHRUN.",https://www.iqrabed.co.in


In [16]:
TEXT_COLUMNS = [
    "institution_name",
    "state",
    "district",
    "location",
    "institution_type",
    "management_type",
    "affiliated_university_name",
    "university_type",
    "administrative_ministry"
]

for column in TEXT_COLUMNS:
    combined_df[column] = combined_df[column].apply(
        lambda x: re.sub(r"\s+", " ", x).strip()
        if isinstance(x, str) else x
    )

print(combined_df[TEXT_COLUMNS].head(10))

                                    institution_name          state district  \
0  061-PT. SRI JAGANNATH PRASAD MAHAVIDHYALAYA,DA...  Uttar Pradesh     Agra   
1  100002-J.D.M.V.P. SAMAJ'S SHRI S. S. PATIL ART...    Maharashtra  Jalgaon   
2  100004-Khandesh College Education Society'S Co...    Maharashtra  Jalgaon   
3    100005-KCES'S S.S.MANIYAR LAW COLLEGE, JALGAON.    Maharashtra  Jalgaon   
4  100006-KRIDA RASIK EDUCATION SOCIETY INSTITUTE...    Maharashtra  Jalgaon   
5  100007-SSBT'S COLLEGE OF ENGINEERING & TECHNOL...    Maharashtra  Jalgaon   
6  100008-SES'S COLLEGE OF PHYSICAL EDUCATION, JA...    Maharashtra  Jalgaon   
7  100009- IQRA EDUCATION SOCIETY'S H.J.THIM COLL...    Maharashtra  Jalgaon   
8  100010-KCES'S INSTITUTE OF MANAGEMENT & RESEAR...    Maharashtra  Jalgaon   
9    100043-IES'S IQRA COLLEGE OF EDUCATION, MEHRUN.    Maharashtra  Jalgaon   

  location    institution_type                   management_type  \
0    Rural  Affiliated College                  Pri

In [17]:
duplicate_codes = combined_df[
    combined_df.duplicated(subset=["aishe_code"], keep=False)
].sort_values("aishe_code")

print("Total duplicate records:", len(duplicate_codes))
print("Unique duplicated AISHE codes:", duplicate_codes["aishe_code"].nunique())

duplicate_codes[
    ["aishe_code", "institution_name", "institution_category", "source_dataset"]
].head(30)

Total duplicate records: 2892
Unique duplicated AISHE codes: 1446


,aishe_code,institution_name,institution_category,source_dataset
53329,C-10216,Vinayaka Mission's College of Pharmacy,Colleges,colleges
74584,C-10216,Vinayrks Mission's College of Pharmacy,Vidya Lakshmi,vidya_lakshmi
8513,C-10240,"College of Basic Sciences and Humanities,Pusa",Colleges,colleges
73489,C-10240,"College of Basic Sciences and Humanities,Pusa",Vidya Lakshmi,vidya_lakshmi
8286,C-10243,"College of Agricultural Engineering,Pusa",Colleges,colleges
73490,C-10243,"College of Agricultural Engineering,Pusa",Vidya Lakshmi,vidya_lakshmi
8528,C-10244,"College of Community Science,Pusa",Colleges,colleges
73491,C-10244,"College of Community Science,Pusa",Vidya Lakshmi,vidya_lakshmi
1702,C-10250,Amar Shaheed Baba Ajit Singh Jujhar Singh Memo...,Colleges,colleges
74364,C-10250,Amar Shaheed Baba Ajit Singh Jujhar Singh Memo...,Vidya Lakshmi,vidya_lakshmi


In [18]:
duplicate_names = combined_df[
    combined_df.duplicated(
        subset=["institution_name"],
        keep=False
    )
].sort_values("institution_name")

print("Duplicate name records:", len(duplicate_names))
print("Unique duplicated names:", duplicate_names["institution_name"].nunique())

duplicate_names[
    ["aishe_code", "institution_name", "institution_category", "source_dataset"]
].head(20)

Duplicate name records: 3147
Unique duplicated names: 1474


,aishe_code,institution_name,institution_category,source_dataset
56411,S-18967,6032-DEPARTMENT OF PHARMACY PUNE,Standalone,standalone
74099,S-18967,6032-DEPARTMENT OF PHARMACY PUNE,Vidya Lakshmi,vidya_lakshmi
56585,S-24095,AAKANKSHA INSTITUTE OF TEACHER EDUCATION,Standalone,standalone
56586,S-23539,AAKANKSHA INSTITUTE OF TEACHER EDUCATION,Standalone,standalone
56665,S-25416,ACADEMIC DIVISION OF NCHMCT,Standalone,standalone
74654,S-25416,ACADEMIC DIVISION OF NCHMCT,Vidya Lakshmi,vidya_lakshmi
417,C-56366,ACADEMY OF TECHNOLOGY MANAGEMENT AND SCIENCE,Colleges,colleges
56670,S-3132,ACADEMY OF TECHNOLOGY MANAGEMENT AND SCIENCE,Standalone,standalone
598,C-66849,ADARSH COLLEGE OF EDUCATION,Colleges,colleges
599,C-52656,ADARSH COLLEGE OF EDUCATION,Colleges,colleges


In [19]:
for name, df in standardized_datasets.items():
    duplicate_count = df["aishe_code"].duplicated().sum()

    print(
        f"{name:<15} | "
        f"Duplicate AISHE codes: {duplicate_count}"
    )

colleges        | Duplicate AISHE codes: 0
universities    | Duplicate AISHE codes: 0
standalone      | Duplicate AISHE codes: 0
research        | Duplicate AISHE codes: 1
vidya_lakshmi   | Duplicate AISHE codes: 0


In [20]:
print("Dataset shape:", combined_df.shape)
print("Total records:", len(combined_df))
print("Unique AISHE codes:", combined_df["aishe_code"].nunique())
print("Missing AISHE codes:", combined_df["aishe_code"].isna().sum())
print("Missing institution names:", combined_df["institution_name"].isna().sum())

Dataset shape: (74844, 15)
Total records: 74844
Unique AISHE codes: 73398
Missing AISHE codes: 0
Missing institution names: 0


In [21]:
def create_searchable_text(row):
    fields = [
        ("Institution", "institution_name"),
        ("AISHE Code", "aishe_code"),
        ("Category", "institution_category"),
        ("State", "state"),
        ("District", "district"),
        ("Website", "website"),
        ("Established", "establishment_year"),
        ("Location", "location"),
        ("Institution Type", "institution_type"),
        ("Management", "management_type"),
        ("Affiliated University", "affiliated_university_name"),
        ("University Type", "university_type"),
        ("Administrative Ministry", "administrative_ministry")
    ]

    parts = []

    for label, column in fields:
        value = row[column]

        if pd.notna(value):
            parts.append(f"{label}: {value}")

    return " | ".join(parts)


combined_df["text"] = combined_df.apply(
    create_searchable_text,
    axis=1
)

combined_df[["aishe_code", "institution_name", "text"]].head(5)

,aishe_code,institution_name,text
0,C-59634,"061-PT. SRI JAGANNATH PRASAD MAHAVIDHYALAYA,DA...",Institution: 061-PT. SRI JAGANNATH PRASAD MAHA...
1,C-8913,100002-J.D.M.V.P. SAMAJ'S SHRI S. S. PATIL ART...,Institution: 100002-J.D.M.V.P. SAMAJ'S SHRI S....
2,C-8922,100004-Khandesh College Education Society'S Co...,Institution: 100004-Khandesh College Education...
3,C-9030,"100005-KCES'S S.S.MANIYAR LAW COLLEGE, JALGAON.",Institution: 100005-KCES'S S.S.MANIYAR LAW COL...
4,C-9025,100006-KRIDA RASIK EDUCATION SOCIETY INSTITUTE...,Institution: 100006-KRIDA RASIK EDUCATION SOCI...


In [22]:
import json

chunks = []

for _, row in combined_df.iterrows():
    metadata = {
        column: (
            None if pd.isna(row[column]) else str(row[column])
        )
        for column in UNIFIED_COLUMNS
    }

    chunk = {
        "chunk_id": str(row["aishe_code"]),
        "text": row["text"],
        "metadata": metadata
    }

    chunks.append(chunk)

print("Total JSON chunks:", len(chunks))
print("\nSample chunk:")
print(json.dumps(chunks[0], indent=4, ensure_ascii=False))

Total JSON chunks: 74844

Sample chunk:
{
    "chunk_id": "C-59634",
    "text": "Institution: 061-PT. SRI JAGANNATH PRASAD MAHAVIDHYALAYA,DABRAI,SAMSHABAD,AGRA | AISHE Code: C-59634 | Category: Colleges | State: Uttar Pradesh | District: Agra | Website: https://WWW.JPCOLLEGE.IN.COM | Established: 2016 | Location: Rural | Institution Type: Affiliated College | Management: Private Un-Aided | Affiliated University: Dr. B. R. Ambedkar University, Agra | University Type: State Public University",
    "metadata": {
        "aishe_code": "C-59634",
        "institution_name": "061-PT. SRI JAGANNATH PRASAD MAHAVIDHYALAYA,DABRAI,SAMSHABAD,AGRA",
        "institution_category": "Colleges",
        "state": "Uttar Pradesh",
        "district": "Agra",
        "website": "https://WWW.JPCOLLEGE.IN.COM",
        "establishment_year": "2016",
        "location": "Rural",
        "institution_type": "Affiliated College",
        "management_type": "Private Un-Aided",
        "affiliated_university_co

In [23]:
for index, chunk in enumerate(chunks):
    chunk["chunk_id"] = f"INST-{index + 1:06d}"

print("Total chunks:", len(chunks))
print("Unique chunk IDs:", len(set(chunk["chunk_id"] for chunk in chunks)))

print("\nSample:")
print(json.dumps(chunks[0], indent=4, ensure_ascii=False))

Total chunks: 74844
Unique chunk IDs: 74844

Sample:
{
    "chunk_id": "INST-000001",
    "text": "Institution: 061-PT. SRI JAGANNATH PRASAD MAHAVIDHYALAYA,DABRAI,SAMSHABAD,AGRA | AISHE Code: C-59634 | Category: Colleges | State: Uttar Pradesh | District: Agra | Website: https://WWW.JPCOLLEGE.IN.COM | Established: 2016 | Location: Rural | Institution Type: Affiliated College | Management: Private Un-Aided | Affiliated University: Dr. B. R. Ambedkar University, Agra | University Type: State Public University",
    "metadata": {
        "aishe_code": "C-59634",
        "institution_name": "061-PT. SRI JAGANNATH PRASAD MAHAVIDHYALAYA,DABRAI,SAMSHABAD,AGRA",
        "institution_category": "Colleges",
        "state": "Uttar Pradesh",
        "district": "Agra",
        "website": "https://WWW.JPCOLLEGE.IN.COM",
        "establishment_year": "2016",
        "location": "Rural",
        "institution_type": "Affiliated College",
        "management_type": "Private Un-Aided",
        "affilia

In [24]:
OUTPUT_FILE = PROCESSED_DIR / "chunks.json"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=4, ensure_ascii=False)

print("Saved successfully!")
print("File:", OUTPUT_FILE)
print("Total chunks:", len(chunks))
print("File size:", round(OUTPUT_FILE.stat().st_size / (1024 ** 2), 2), "MB")

Saved successfully!
File: E:\Rasengan\College-Knowledge-Search\data\processed\chunks.json
Total chunks: 74844
File size: 86.65 MB


In [25]:
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    loaded_chunks = json.load(f)

print("Loaded chunks:", len(loaded_chunks))
print("Unique chunk IDs:", len(set(c["chunk_id"] for c in loaded_chunks)))
print("Required keys:", list(loaded_chunks[0].keys()))

assert len(loaded_chunks) == 74844
assert len(set(c["chunk_id"] for c in loaded_chunks)) == 74844
assert all(
    {"chunk_id", "text", "metadata"}.issubset(chunk.keys())
    for chunk in loaded_chunks
)

print("\nJSON validation passed!")

Loaded chunks: 74844
Unique chunk IDs: 74844
Required keys: ['chunk_id', 'text', 'metadata']

JSON validation passed!


## Preprocessing Summary

### 1. Dataset Overview

Five AISHE datasets were used:

| Dataset | Records |
|---|---:|
| Colleges | 54,571 |
| Universities | 1,438 |
| Standalone | 17,092 |
| Research & Development | 291 |
| Vidya Lakshmi | 1,452 |
| **Total** | **74,844** |

### 2. Key Findings

- All five datasets contained different column structures.
- Standardized all datasets into a common schema of 15 columns.
- Found missing values and placeholders such as `-`, `NA`, and empty strings.
- Identified duplicate AISHE codes across datasets and retained all records.
- Found 73,398 unique AISHE codes among 74,844 records.
- No records had missing AISHE codes or institution names.

### 3. Preprocessing Performed

- Loaded and combined all five datasets.
- Standardized column names and schemas.
- Converted placeholder values into `NaN`.
- Cleaned establishment years and converted them into nullable integers.
- Normalized website URLs by adding missing protocols.
- Removed unnecessary whitespace from text fields.
- Generated searchable text representations for each institution.
- Preserved structured metadata for every record.

### 4. JSON Chunk Generation

Each institution was converted into an individual JSON chunk containing:

- `chunk_id`: Unique identifier for each chunk.
- `text`: Searchable textual representation.
- `metadata`: Structured institutional information.

Record-level chunking was used instead of fixed-size chunking because each row already represents an individual institution.

### 5. Final Output

| Metric | Result |
|---|---:|
| Total records | 74,844 |
| Total JSON chunks | 74,844 |
| Unique chunk IDs | 74,844 |
| Unique AISHE codes | 73,398 |
| JSON file size | 86.65 MB |
| Validation | Passed |

**Output file:** `data/processed/chunks.json`

### 6. Conclusion

The five AISHE datasets were successfully cleaned, standardized, combined, and converted into JSON chunks. The generated dataset preserves institutional information and is ready for the next phase: BM25 indexing, embedding generation, FAISS indexing, and hybrid retrieval.